# Agentic AI RAG Solution with LangGraph + ChromaDB

This notebook implements an **Agentic RAG** pipeline that:
1. Loads and chunks a PDF document (World Mental Health Report)
2. Stores chunks in ChromaDB (vector database)
3. Uses semantic search to retrieve relevant context
4. Uses LangGraph to orchestrate an agent that answers questions from the PDF
5. Returns "this document does not have that info" when no meaningful context is found

## Cell 1 — Imports and Environment Setup

In [15]:
# ── Standard library imports ──
import os                                          # Access environment variables
import json                                        # Save and load the vector index manifest
import hashlib                                     # Build stable chunk IDs from text content
from copy import deepcopy                           # Clone manifests before updating them
from typing import Annotated, TypedDict, List      # Type hints for state schema

# ── Environment / secrets ──
from dotenv import load_dotenv                     # Load .env file with API keys
load_dotenv()                                      # Read .env into os.environ

# ── PDF parsing ──
import pdfplumber                                  # Extract text from PDF pages

# ── LangChain core ──
from langchain_core.documents import Document      # Document wrapper for chunks
from langchain_core.messages import (
    HumanMessage,                                  # User question message
    AIMessage,                                     # Assistant response message
    SystemMessage,                                 # System prompt message
)
from langchain_core.prompts import ChatPromptTemplate  # Build chat prompts

# ── Text splitting ──
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,                # Smart chunking by paragraphs/sentences
)

# ── Embeddings and LLM ──
from langchain_openai import (
    OpenAIEmbeddings,                              # text-embedding-3-small for vectors
    ChatOpenAI,                                    # GPT model for answering
)

# ── ChromaDB vector store ──
from langchain_chroma import Chroma                # ChromaDB integration

# ── LangGraph ──
from langgraph.graph import START, END, StateGraph  # Graph building blocks

print("All imports loaded successfully.")

All imports loaded successfully.


## Cell 2 — Configuration

In [16]:
# ── Path to the PDF document ──
PDF_PATH = "/Users/vinothkumarkothandapani/Downloads/AIProject/MyProject/langgraph/The_World_Mental_Health_Report_transforming_mental.pdf"

# ── ChromaDB collection name (versioned so we can introduce stable IDs safely) ──
COLLECTION_NAME = "world_mental_health_report_v2"

# ── ChromaDB persistent storage directory ──
CHROMA_DIR = "/Users/vinothkumarkothandapani/Downloads/AIProject/MyProject/langgraph/chroma_db"

# ── Sidecar manifest file to track chunk IDs per page for future updates ──
VECTOR_INDEX_MANIFEST_PATH = os.path.join(CHROMA_DIR, "vector_index_manifest.json")

# ── Chunking parameters ──
CHUNK_SIZE = 1000       # Maximum characters per chunk
CHUNK_OVERLAP = 200     # Overlap between consecutive chunks for context continuity

# ── Number of chunks to retrieve per query ──
TOP_K = 5

# ── Relevance score threshold (lower = more similar, for cosine distance) ──
# Chunks with score above this are considered irrelevant
RELEVANCE_THRESHOLD = 1.2

# ── LLM model name ──
LLM_MODEL = "gpt-4o-mini"

# ── Verify the PDF file exists ──
assert os.path.isfile(PDF_PATH), f"PDF not found at: {PDF_PATH}"
print(f"PDF found: {PDF_PATH}")
print(f"ChromaDB will persist to: {CHROMA_DIR}")
print(f"Vector index manifest: {VECTOR_INDEX_MANIFEST_PATH}")

PDF found: /Users/vinothkumarkothandapani/Downloads/AIProject/MyProject/langgraph/The_World_Mental_Health_Report_transforming_mental.pdf
ChromaDB will persist to: /Users/vinothkumarkothandapani/Downloads/AIProject/MyProject/langgraph/chroma_db
Vector index manifest: /Users/vinothkumarkothandapani/Downloads/AIProject/MyProject/langgraph/chroma_db/vector_index_manifest.json


## Cell 3 — Load PDF and Extract Text

In [17]:
def load_pdf(pdf_path: str) -> List[Document]:
    """
    Open the PDF with pdfplumber and convert each page into a
    LangChain Document object.  Page number is stored in metadata
    so we can cite it later.
    """
    documents = []                                          # Will hold one Document per page

    with pdfplumber.open(pdf_path) as pdf:                  # Open the PDF file
        total_pages = len(pdf.pages)                        # Count total pages
        print(f"PDF has {total_pages} pages.")              # Progress feedback

        for page_num, page in enumerate(pdf.pages, start=1):  # Iterate pages (1-indexed)
            text = page.extract_text()                      # Extract raw text from the page

            if text and text.strip():                       # Skip blank pages
                doc = Document(
                    page_content=text.strip(),              # Store cleaned text
                    metadata={                              # Attach page metadata
                        "source": pdf_path,
                        "page": page_num,
                    },
                )
                documents.append(doc)

    print(f"Extracted text from {len(documents)} non-empty pages.")
    return documents

# ── Run the loader ──
raw_documents = load_pdf(PDF_PATH)

# ── Preview first page text (first 500 chars) ──
print("\n--- Preview of page 1 ---")
print(raw_documents[0].page_content[:500])

PDF has 2 pages.
Extracted text from 2 non-empty pages.

--- Preview of page 1 ---
service delivery. Peer-led service providers have an advantage in ganizations, it should be well noted that authentic and meaningful
comparison with other professional services, through having lived inclusion can only happen when these persons are involved from
experience and practical knowledge of navigating mental health the very start and not as an afterthought. At the same time, it is
related services and processes, and therefore being in a better po- critical to consider diversity (gender, 


## Cell 4 — Chunk the Documents

In [18]:
def chunk_documents(documents: List[Document], chunk_size: int, chunk_overlap: int) -> List[Document]:
    """
    Split large page-level documents into smaller, overlapping chunks.
    RecursiveCharacterTextSplitter tries to split on paragraph boundaries
    first, then sentences, then words, which preserves semantic coherence.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,                              # Maximum characters per chunk
        chunk_overlap=chunk_overlap,                        # Overlap for continuity
        separators=["\n\n", "\n", ". ", " ", ""],          # Split hierarchy
        length_function=len,                                # Use character count
    )

    chunks = splitter.split_documents(documents)            # Perform the split
    print(f"Created {len(chunks)} chunks from {len(documents)} pages.")
    return chunks


def build_chunk_id(source_path: str, page_number: int, chunk_index: int, chunk_text: str) -> str:
    """
    Create a stable chunk ID so we can delete and refresh specific pages later.
    The ID includes the source file, page number, chunk index, and a text hash.
    """
    text_hash = hashlib.sha1(chunk_text.encode("utf-8")).hexdigest()[:12]  # Short stable fingerprint
    source_name = os.path.basename(source_path)                              # Keep the ID readable
    return f"{source_name}::p{page_number:04d}::c{chunk_index:04d}::{text_hash}"


def index_chunks_with_ids(documents: List[Document]) -> tuple[list[Document], dict]:
    """
    Attach stable IDs and metadata to every chunk.
    The returned manifest records which chunk IDs belong to each page.
    """
    indexed_documents = []
    manifest = {
        "source": PDF_PATH,
        "collection": COLLECTION_NAME,
        "pages": {},
    }

    for chunk_index, doc in enumerate(documents):
        page_number = int(doc.metadata.get("page", -1))
        chunk_text = doc.page_content.strip()
        chunk_id = build_chunk_id(PDF_PATH, page_number, chunk_index, chunk_text)

        chunk_metadata = dict(doc.metadata)                                   # Copy existing page metadata
        chunk_metadata["chunk_index"] = chunk_index                          # Store the chunk order
        chunk_metadata["chunk_id"] = chunk_id                                # Store the stable chunk ID
        chunk_metadata["document_version"] = 2                               # Mark this as the updateable version

        indexed_doc = Document(page_content=chunk_text, metadata=chunk_metadata)
        indexed_documents.append(indexed_doc)
        manifest["pages"].setdefault(str(page_number), []).append(chunk_id)

    manifest["total_chunks"] = len(indexed_documents)
    return indexed_documents, manifest


def save_vector_index_manifest(manifest: dict, manifest_path: str = VECTOR_INDEX_MANIFEST_PATH) -> None:
    """Persist the page-to-chunk mapping so we can refresh pages later."""
    os.makedirs(os.path.dirname(manifest_path), exist_ok=True)              # Ensure the target directory exists
    with open(manifest_path, "w", encoding="utf-8") as file:
        json.dump(manifest, file, indent=2)


def load_vector_index_manifest(manifest_path: str = VECTOR_INDEX_MANIFEST_PATH) -> dict | None:
    """Load the existing manifest if one has already been written to disk."""
    if not os.path.exists(manifest_path):
        return None
    with open(manifest_path, "r", encoding="utf-8") as file:
        return json.load(file)


# ── Run the chunker ──
chunks = chunk_documents(raw_documents, CHUNK_SIZE, CHUNK_OVERLAP)

# ── Add stable IDs so specific pages can be updated later without rebuilding everything ──
indexed_chunks, vector_index_manifest = index_chunks_with_ids(chunks)
save_vector_index_manifest(vector_index_manifest)

# ── Preview a sample chunk ──
print(f"\n--- Sample chunk (index 3) ---")
print(f"Page: {indexed_chunks[3].metadata['page']}")
print(f"Chunk ID: {indexed_chunks[3].metadata['chunk_id']}")
print(indexed_chunks[3].page_content[:300])

Created 17 chunks from 2 pages.

--- Sample chunk (index 3) ---
Page: 1
Chunk ID: The_World_Mental_Health_Report_transforming_mental.pdf::p0001::c0003::4a6334c12362
the planning to the implementation phase of all new develop-
new understanding, new hope. Geneva: World Health Organization, 2001.
ments in the mental health field. Equally important is for people 3. Thornicroft G, Tansella M. Epidemiol Psichiatr Soc 2005:14:1-3.
with lived experience to be integrat


## Cell 5 — Build ChromaDB Vector Store

In [19]:
def build_vector_store(
    chunks: List[Document],
    collection_name: str,
    persist_dir: str,
) -> Chroma:
    """
    Embed all chunks using OpenAI embeddings and store them in a
    persistent ChromaDB collection. The chunk IDs are deterministic,
    so future page refreshes can delete and replace exact pages.
    """
    # ── Create the embedding model ──
    embeddings = OpenAIEmbeddings(
        model="text-embedding-3-small",                     # Fast, cost-effective embeddings
    )

    # ── Extract stable IDs from the chunk metadata ──
    chunk_ids = [doc.metadata["chunk_id"] for doc in chunks]  # Use deterministic IDs for every chunk

    # ── Load an existing collection if one is already available ──
    vector_store = Chroma(
        collection_name=collection_name,
        embedding_function=embeddings,
        persist_directory=persist_dir,
    )

    current_count = vector_store._collection.count()           # How many vectors are already stored
    if current_count == 0:
        print(f"Creating new ChromaDB collection at: {persist_dir}")
        vector_store = Chroma.from_documents(
            documents=chunks,                                 # The chunked documents
            embedding=embeddings,                             # Embedding model
            ids=chunk_ids,                                    # Deterministic chunk IDs
            collection_name=collection_name,                  # Collection identifier
            persist_directory=persist_dir,                    # Where to save on disk
        )
        current_count = vector_store._collection.count()
        print(f"Stored {current_count} vectors in collection '{collection_name}'.")
    else:
        print(f"Loaded existing ChromaDB collection '{collection_name}' with {current_count} vectors.")

    return vector_store


# ── Build or load the vector store ──
vector_store = build_vector_store(indexed_chunks, COLLECTION_NAME, CHROMA_DIR)

Creating new ChromaDB collection at: /Users/vinothkumarkothandapani/Downloads/AIProject/MyProject/langgraph/chroma_db
Stored 17 vectors in collection 'world_mental_health_report_v2'.


## Cell 5.5 — Page Refresh Logic for Outdated PDF Content

In [20]:
def refresh_pages_in_vector_store(
    store: Chroma,
    manifest: dict,
    page_updates: dict[int, str],
    manifest_path: str = VECTOR_INDEX_MANIFEST_PATH,
) -> dict:
    """
    Replace only the specified pages inside the vector database.
    This keeps the rest of the index untouched and avoids a full rebuild.
    """
    updated_manifest = deepcopy(manifest)                                # Work on a copy so callers keep the original manifest
    splitter = RecursiveCharacterTextSplitter(                           # Use the same chunking strategy as the main pipeline
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],
        length_function=len,
    )

    for page_number, updated_text in page_updates.items():
        # ── Delete the stale chunks that belong to this page ──
        stale_ids = updated_manifest["pages"].get(str(page_number), [])
        if stale_ids:
            store.delete(ids=stale_ids)

        # ── Rebuild the page text into fresh chunks ──
        updated_page_document = Document(
            page_content=updated_text.strip(),
            metadata={
                "source": PDF_PATH,
                "page": page_number,
            },
        )
        updated_chunks = splitter.split_documents([updated_page_document])
        indexed_updated_chunks, page_manifest = index_chunks_with_ids(updated_chunks)
        new_ids = [doc.metadata["chunk_id"] for doc in indexed_updated_chunks]

        # ── Insert the fresh chunks back into ChromaDB ──
        store.add_documents(indexed_updated_chunks, ids=new_ids)

        # ── Update the manifest so future refreshes know the new chunk IDs ──
        updated_manifest["pages"][str(page_number)] = page_manifest["pages"][str(page_number)]

    updated_manifest["total_chunks"] = sum(len(ids) for ids in updated_manifest["pages"].values())
    save_vector_index_manifest(updated_manifest, manifest_path=manifest_path)
    return updated_manifest


def preview_manifest(manifest: dict) -> None:
    """Print a compact summary of the page-to-chunk mapping."""
    print(f"Manifest source: {manifest['source']}")
    print(f"Manifest collection: {manifest['collection']}")
    print(f"Manifest total chunks: {manifest['total_chunks']}")
    for page, ids in manifest["pages"].items():
        print(f"  Page {page}: {len(ids)} chunk IDs")

## Cell 6 — Create the Semantic Retriever

In [21]:
def create_retriever(vector_store: Chroma, top_k: int):
    """
    Build a retriever that uses similarity_score_threshold search.
    This returns the top-k most similar chunks along with their
    relevance scores so we can filter out weak matches.
    """
    retriever = vector_store.as_retriever(
        search_type="similarity",                           # Cosine similarity search
        search_kwargs={"k": top_k},                         # Return top_k results
    )
    print(f"Retriever ready — will return top {top_k} chunks per query.")
    return retriever

# ── Create the retriever ──
retriever = create_retriever(vector_store, TOP_K)

# ── Quick sanity check: run a test retrieval ──
test_docs = retriever.invoke("What is mental health?")
print(f"\nTest retrieval returned {len(test_docs)} chunks.")
for i, doc in enumerate(test_docs, 1):
    print(f"  Chunk {i} (page {doc.metadata['page']}): {doc.page_content[:80]}...")

Retriever ready — will return top 5 chunks per query.

Test retrieval returned 5 chunks.
  Chunk 1 (page 1): nizing country and cultural differences, it provides clear pointers receive far ...
  Chunk 2 (page 2): interdependent pillars: mental health value, changing environ- Health Report on ...
  Chunk 3 (page 2): neglect. The report argues that individuals, families, communi- that are able to...
  Chunk 4 (page 2): that are critical to improved mental health but that fall within the mental heal...
  Chunk 5 (page 1): Going forward, for governments to truly commit to the inclu-
DOI:10.1002/wps.210...


## Cell 7 — Define the LLM and Prompt

In [22]:
# ── Initialise the chat model ──
llm = ChatOpenAI(
    model=LLM_MODEL,                                       # Use GPT-4o-mini for speed + quality
    temperature=0,                                         # Deterministic answers
)

# ── RAG prompt template ──
# The system message tells the LLM to answer ONLY from the provided context.
# If the context does not contain the answer, it must say so explicitly.
RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     """You are a helpful assistant that answers questions based ONLY on the provided context from a PDF document.

RULES:
1. Answer the question using ONLY the context provided below.
2. If the context does not contain enough information to answer the question, respond with:
   "This document does not have that info."
3. Always cite the page number(s) where you found the information.
4. Be concise but thorough.

CONTEXT:
{context}
"""),
    ("human", "{question}"),
])

print("LLM and prompt template configured.")

LLM and prompt template configured.


## Cell 8 — Define the LangGraph Agent State and Nodes

In [23]:
# ── Define the state schema that flows through the graph ──
class AgentState(TypedDict):
    """State passed between graph nodes."""
    question: str                          # The user's original question
    retrieved_docs: List[Document]          # Chunks returned by the retriever
    context: str                           # Formatted context string for the LLM
    answer: str                            # Final answer from the LLM
    has_context: bool                      # Whether meaningful context was found


# ──────────────────────────────────────────────
# Node 1: RETRIEVE — fetch relevant chunks
# ──────────────────────────────────────────────
def retrieve_node(state: AgentState) -> dict:
    """
    Use the semantic retriever to find the most relevant chunks
    for the user's question.
    """
    question = state["question"]                           # Get the user question
    docs = retriever.invoke(question)                      # Semantic search in ChromaDB
    print(f"[Retrieve] Found {len(docs)} chunks for: '{question[:60]}...'")
    return {"retrieved_docs": docs}                        # Pass docs to next node


# ──────────────────────────────────────────────
# Node 2: GRADE — check if retrieved context is relevant
# ──────────────────────────────────────────────
def grade_node(state: AgentState) -> dict:
    """
    Check whether the retrieved documents contain meaningful content.
    Uses similarity_search_with_score to get distance scores and
    filters out chunks that are too dissimilar.
    """
    question = state["question"]
    # ── Get scores along with documents ──
    scored_results = vector_store.similarity_search_with_score(question, k=TOP_K)

    relevant_docs = []                                     # Will hold docs that pass the filter
    for doc, score in scored_results:
        # Lower score = more similar (cosine distance)
        if score <= RELEVANCE_THRESHOLD:                   # Only keep relevant chunks
            relevant_docs.append(doc)

    if relevant_docs:
        # ── Build a formatted context string with page citations ──
        context_parts = []
        for doc in relevant_docs:
            page = doc.metadata.get("page", "?")
            context_parts.append(f"[Page {page}]\n{doc.page_content}")
        context_str = "\n\n---\n\n".join(context_parts)    # Separate chunks clearly

        print(f"[Grade] {len(relevant_docs)} chunks passed relevance filter.")
        return {
            "retrieved_docs": relevant_docs,
            "context": context_str,
            "has_context": True,
        }
    else:
        print("[Grade] No relevant chunks found — question is out of scope.")
        return {
            "retrieved_docs": [],
            "context": "",
            "has_context": False,
        }


# ──────────────────────────────────────────────
# Node 3: GENERATE — produce the answer
# ──────────────────────────────────────────────
def generate_node(state: AgentState) -> dict:
    """
    If relevant context was found, ask the LLM to answer using that context.
    Otherwise, return a polite 'no info' message without calling the LLM.
    """
    if not state.get("has_context", False):
        # ── No meaningful context — skip LLM call ──
        answer = "This document does not have that info."
        print(f"[Generate] No context → returning fallback answer.")
    else:
        # ── Build the prompt with context + question ──
        prompt_value = RAG_PROMPT.invoke({
            "context": state["context"],
            "question": state["question"],
        })
        # ── Call the LLM ──
        response = llm.invoke(prompt_value)
        answer = response.content                          # Extract the text answer
        print(f"[Generate] LLM answered ({len(answer)} chars).")

    return {"answer": answer}


print("Agent nodes defined: retrieve_node, grade_node, generate_node")

Agent nodes defined: retrieve_node, grade_node, generate_node


## Cell 9 — Build and Compile the LangGraph

In [24]:
# ── Create the state graph ──
workflow = StateGraph(AgentState)                          # Initialise with our state schema

# ── Add nodes (each node is a step in the RAG pipeline) ──
workflow.add_node("retrieve", retrieve_node)               # Step 1: Semantic retrieval
workflow.add_node("grade", grade_node)                     # Step 2: Relevance grading
workflow.add_node("generate", generate_node)               # Step 3: Answer generation

# ── Define edges (the flow of execution) ──
workflow.add_edge(START, "retrieve")                       # Entry point → retrieve
workflow.add_edge("retrieve", "grade")                     # retrieve → grade
workflow.add_edge("grade", "generate")                     # grade → generate
workflow.add_edge("generate", END)                         # generate → exit

# ── Compile the graph into a runnable ──
app = workflow.compile()                                   # Compiled graph ready to invoke

print("LangGraph compiled successfully.")
print("Flow: START → retrieve → grade → generate → END")

LangGraph compiled successfully.
Flow: START → retrieve → grade → generate → END


## Cell 10 — Helper Function to Query the Agent

In [25]:
def ask(question: str) -> str:
    """
    Send a question through the RAG agent graph and return the answer.
    Prints the full execution trace for debugging.
    """
    print(f"\n{'='*60}")
    print(f"QUESTION: {question}")
    print(f"{'='*60}")

    # ── Invoke the compiled graph with the initial state ──
    result = app.invoke({
        "question": question,                              # Pass the user query
        "retrieved_docs": [],                              # Initialise empty
        "context": "",                                     # Initialise empty
        "answer": "",                                      # Initialise empty
        "has_context": False,                              # Initialise as False
    })

    answer = result["answer"]                               # Extract the final answer
    print(f"\nANSWER:\n{answer}")
    print(f"{'='*60}\n")
    return answer

print("ask() helper ready. Usage: ask('Your question here')")

ask() helper ready. Usage: ask('Your question here')


## Cell 11 — Test: Questions WITH context in the document

In [26]:
# ── Test 1: A question the document should be able to answer ──
ask("What is mental health according to this report?")


QUESTION: What is mental health according to this report?
[Retrieve] Found 5 chunks for: 'What is mental health according to this report?...'
[Grade] 5 chunks passed relevance filter.
[Generate] LLM answered (233 chars).

ANSWER:
Mental health is defined in the report as “A state of mental well-being that enables people to cope with the stresses of life, to realize their abilities, to learn well and work well, and to contribute to their communities” (Page 1).



'Mental health is defined in the report as “A state of mental well-being that enables people to cope with the stresses of life, to realize their abilities, to learn well and work well, and to contribute to their communities” (Page 1).'

In [27]:
# ── Test 2: Another in-scope question ──
ask("What are the risk factors for mental health conditions?")


QUESTION: What are the risk factors for mental health conditions?
[Retrieve] Found 5 chunks for: 'What are the risk factors for mental health conditions?...'
[Grade] 5 chunks passed relevance filter.
[Generate] LLM answered (38 chars).

ANSWER:
This document does not have that info.



'This document does not have that info.'

## Cell 12 — Test: Questions WITHOUT context in the document

In [28]:
# ── Test 3: Completely off-topic — should return "no info" ──
ask("What is the capital of Mars?")


QUESTION: What is the capital of Mars?
[Retrieve] Found 5 chunks for: 'What is the capital of Mars?...'
[Grade] No relevant chunks found — question is out of scope.
[Generate] No context → returning fallback answer.

ANSWER:
This document does not have that info.



'This document does not have that info.'

In [29]:
# ── Test 4: Related to health but not in this document ──
ask("What are the side effects of paracetamol?")


QUESTION: What are the side effects of paracetamol?
[Retrieve] Found 5 chunks for: 'What are the side effects of paracetamol?...'
[Grade] No relevant chunks found — question is out of scope.
[Generate] No context → returning fallback answer.

ANSWER:
This document does not have that info.



'This document does not have that info.'

## Cell 13 — Demo: Refresh Only One Outdated Page

In [30]:
# ── Create a temporary demo collection so the refresh test does not disturb the main index ──
demo_collection_name = f"{COLLECTION_NAME}_demo_refresh"            # Separate collection for the update demonstration
demo_manifest_path = os.path.join(CHROMA_DIR, "vector_index_manifest_demo.json")  # Separate manifest file for the demo

demo_embeddings = OpenAIEmbeddings(                                  # Same embedding model as the main index
    model="text-embedding-3-small",
)

demo_store = Chroma(                                                 # Load the demo collection if it already exists
    collection_name=demo_collection_name,
    embedding_function=demo_embeddings,
    persist_directory=CHROMA_DIR,
)

if demo_store._collection.count() == 0:                              # Build the demo collection only once
    demo_store = Chroma.from_documents(
        documents=indexed_chunks,                                    # Reuse the already chunked documents
        embedding=demo_embeddings,
        ids=[doc.metadata["chunk_id"] for doc in indexed_chunks],    # Reuse stable chunk IDs
        collection_name=demo_collection_name,
        persist_directory=CHROMA_DIR,
    )

print(f"Demo collection before refresh: {demo_store._collection.count()} vectors")

demo_manifest = deepcopy(vector_index_manifest)                      # Clone the main manifest so the test stays isolated
demo_manifest["collection"] = demo_collection_name                  # Record the demo collection name

# ── Simulate a revised page text to prove that stale chunks can be replaced in place ──
updated_page_2_text = raw_documents[1].page_content + "\n\nThis refreshed page now mentions crisis care continuity and updated support pathways."

# ── Refresh only page 2 inside the demo collection ──
demo_manifest = refresh_pages_in_vector_store(
    store=demo_store,
    manifest=demo_manifest,
    page_updates={2: updated_page_2_text},
    manifest_path=demo_manifest_path,
)

print(f"Demo collection after refresh: {demo_store._collection.count()} vectors")
preview_manifest(demo_manifest)

# ── Verify that the refreshed page can now be retrieved semantically ──
demo_query = "crisis care continuity"
demo_hits = demo_store.similarity_search(demo_query, k=3)
print(f"\nDemo retrieval for: {demo_query}")
for idx, hit in enumerate(demo_hits, 1):
    print(f"  Hit {idx} | page {hit.metadata.get('page')} | {hit.page_content[:120]}...")

Demo collection before refresh: 17 vectors
Demo collection after refresh: 18 vectors
Manifest source: /Users/vinothkumarkothandapani/Downloads/AIProject/MyProject/langgraph/The_World_Mental_Health_Report_transforming_mental.pdf
Manifest collection: world_mental_health_report_v2_demo_refresh
Manifest total chunks: 18
  Page 1: 8 chunk IDs
  Page 2: 10 chunk IDs

Demo retrieval for: crisis care continuity
  Hit 1 | page 2 | This refreshed page now mentions crisis care continuity and updated support pathways....
  Hit 2 | page 2 | require a full read of the report. At the centre of the services ap- DOI:10.1002/wps.21018
proach is community-based men...
  Hit 3 | page 2 | time when burden of disease, rather than mortality alone, was be- mental health care that is provided outside of a psych...
